In [1]:
# ============================================================
# NOTEBOOK 16 — FINAL DATASET AUDIT
# CELL 1 — IMPORTS AND PATHS
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd


PROJECT_ROOT = Path(
    r"D:\Pancreatic_Cancer_Thesis"
)

DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DIR = (
    DATA_DIR / "processed"
)

IMAGES_DIR = (
    PROCESSED_DIR / "images"
)

MASKS_DIR = (
    PROCESSED_DIR / "masks"
)

METADATA_FILE = (
    PROCESSED_DIR / "metadata.csv"
)

print("=" * 70)
print("NOTEBOOK 16 — FINAL DATASET AUDIT")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nMetadata:")
print(METADATA_FILE)

print("\nProcessed images:")
print(IMAGES_DIR)

print("\nProcessed masks:")
print(MASKS_DIR)

NOTEBOOK 16 — FINAL DATASET AUDIT

Project root:
D:\Pancreatic_Cancer_Thesis

Metadata:
D:\Pancreatic_Cancer_Thesis\data\processed\metadata.csv

Processed images:
D:\Pancreatic_Cancer_Thesis\data\processed\images

Processed masks:
D:\Pancreatic_Cancer_Thesis\data\processed\masks


In [2]:
# ============================================================
# CELL 2 — LOAD METADATA
# ============================================================

if not METADATA_FILE.exists():
    raise FileNotFoundError(
        f"Metadata file not found:\n{METADATA_FILE}"
    )

metadata = pd.read_csv(
    METADATA_FILE
)

print("=" * 70)
print("METADATA")
print("=" * 70)

print(
    "Rows:",
    len(metadata)
)

print(
    "Columns:",
    len(metadata.columns)
)

print("\nColumns:")
for column in metadata.columns:
    print(" ", column)

print("\nFirst 5 rows:")
display(metadata.head())

METADATA
Rows: 2238
Columns: 18

Columns:
  study_id
  label
  image_path
  mask_path
  image_shape
  mask_shape
  image_dtype
  mask_dtype
  mask_labels
  patient_id
  patient_age
  patient_sex
  scanner
  diagnosis
  diagnosis_source
  roi_size
  target_spacing
  hu_window

First 5 rows:


,study_id,label,image_path,mask_path,image_shape,mask_shape,image_dtype,mask_dtype,mask_labels,patient_id,patient_age,patient_sex,scanner,diagnosis,diagnosis_source,roi_size,target_spacing,hu_window
0,100000_00001,NaN,images\100000_00001.npy,masks\100000_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,2,3,4,6",100000,42.0,F,TOSHIBA,non-PDAC,radiology,"128,160,192","1.0,1.0,3.0","-150,250"
1,100001_00001,NaN,images\100001_00001.npy,masks\100001_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,2,3,4,5,6",100001,68.0,M,SIEMENS,non-PDAC,radiology,"128,160,192","1.0,1.0,3.0","-150,250"
2,100002_00001,NaN,images\100002_00001.npy,masks\100002_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,1,2,3,4,5,6",100002,77.0,F,TOSHIBA,PDAC,pathology,"128,160,192","1.0,1.0,3.0","-150,250"
3,100003_00001,NaN,images\100003_00001.npy,masks\100003_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,1,2,3,4,5,6",100003,57.0,M,TOSHIBA,PDAC,cytology,"128,160,192","1.0,1.0,3.0","-150,250"
4,100004_00001,NaN,images\100004_00001.npy,masks\100004_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,2,3,4,5,6",100004,68.0,M,SIEMENS,non-PDAC,radiology,"128,160,192","1.0,1.0,3.0","-150,250"


In [3]:
# ============================================================
# CELL 3 — METADATA INTEGRITY
# ============================================================

print("=" * 70)
print("METADATA INTEGRITY")
print("=" * 70)

EXPECTED_CASES = 2238

assert len(metadata) == EXPECTED_CASES, (
    f"Expected {EXPECTED_CASES} metadata rows, "
    f"found {len(metadata)}."
)

# ------------------------------------------------------------
# Required identifiers
# ------------------------------------------------------------

if "study_id" not in metadata.columns:
    raise ValueError(
        "study_id column is missing."
    )

study_ids = (
    metadata["study_id"]
    .astype(str)
    .str.strip()
)

print(
    "Unique study IDs:",
    study_ids.nunique()
)

duplicate_ids = study_ids[
    study_ids.duplicated(keep=False)
]

print(
    "Duplicate study IDs:",
    len(duplicate_ids)
)

assert study_ids.nunique() == EXPECTED_CASES
assert len(duplicate_ids) == 0

# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

missing_counts = (
    metadata.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nColumns with missing values:")

display(
    missing_counts[
        missing_counts > 0
    ]
)

print("\n✓ Study IDs are unique.")

METADATA INTEGRITY
Unique study IDs: 2238
Duplicate study IDs: 0

Columns with missing values:


label          2238
scanner         276
patient_sex     274
patient_age     274
dtype: int64


✓ Study IDs are unique.


In [4]:
# ============================================================
# CELL 4 — TECHNICAL METADATA COMPLETENESS
# ============================================================

print("=" * 70)
print("TECHNICAL METADATA COMPLETENESS")
print("=" * 70)

technical_columns = [
    "study_id",
    "image_path",
    "mask_path",
    "image_shape",
    "mask_shape",
    "image_dtype",
    "mask_dtype",
    "mask_labels",
    "roi_size",
    "target_spacing",
    "hu_window",
]

for column in technical_columns:

    if column not in metadata.columns:
        print(
            f"{column:<20}: COLUMN MISSING"
        )
        continue

    missing = int(
        metadata[column].isna().sum()
    )

    print(
        f"{column:<20}: missing = {missing}"
    )

TECHNICAL METADATA COMPLETENESS
study_id            : missing = 0
image_path          : missing = 0
mask_path           : missing = 0
image_shape         : missing = 0
mask_shape          : missing = 0
image_dtype         : missing = 0
mask_dtype          : missing = 0
mask_labels         : missing = 0
roi_size            : missing = 0
target_spacing      : missing = 0
hu_window           : missing = 0


In [5]:
# ============================================================
# CELL 5 — PATIENT-LEVEL AUDIT
# ============================================================

print("=" * 70)
print("PATIENT-LEVEL AUDIT")
print("=" * 70)

if "patient_id" not in metadata.columns:
    raise ValueError(
        "patient_id column is missing from metadata."
    )

patient_ids = (
    metadata["patient_id"]
    .astype(str)
    .str.strip()
)

unique_patients = (
    patient_ids.nunique()
)

print(
    "Total cases          :",
    len(metadata)
)

print(
    "Unique patients      :",
    unique_patients
)

print(
    "Cases per patient    :",
    len(metadata) / unique_patients
)

PATIENT-LEVEL AUDIT
Total cases          : 2238
Unique patients      : 2224
Cases per patient    : 1.006294964028777


In [6]:
# ============================================================
# CELL 6 — CASES PER PATIENT
# ============================================================

cases_per_patient = (
    metadata
    .groupby("patient_id")
    .size()
    .sort_values(ascending=False)
)

print("=" * 70)
print("CASES PER PATIENT")
print("=" * 70)

print(
    "Patients:",
    len(cases_per_patient)
)

print(
    "Patients with >1 case:",
    int(
        (cases_per_patient > 1)
        .sum()
    )
)

print(
    "Maximum cases for one patient:",
    int(
        cases_per_patient.max()
    )
)

print("\nDistribution:")
print(
    cases_per_patient.value_counts()
    .sort_index()
)

print("\nPatients with multiple cases:")
display(
    cases_per_patient[
        cases_per_patient > 1
    ].head(50)
)

CASES PER PATIENT
Patients: 2224
Patients with >1 case: 11
Maximum cases for one patient: 5

Distribution:
1    2213
2      10
5       1
Name: count, dtype: int64

Patients with multiple cases:


patient_id
100047    5
100265    2
100455    2
100416    2
100268    2
100417    2
100456    2
101361    2
101595    2
101970    2
102065    2
dtype: int64

In [7]:
# ============================================================
# CELL 7 — STUDY/PATIENT RELATIONSHIP
# ============================================================

patient_case_summary = (
    metadata
    .groupby("patient_id")
    .agg(
        n_cases=("study_id", "nunique"),
        studies=("study_id", "count"),
    )
    .sort_values(
        "n_cases",
        ascending=False
    )
)

print("=" * 70)
print("STUDY ↔ PATIENT RELATIONSHIP")
print("=" * 70)

print(
    "Unique patients:",
    patient_case_summary.shape[0]
)

print(
    "Patients with multiple studies:",
    int(
        (
            patient_case_summary["n_cases"]
            > 1
        ).sum()
    )
)

display(
    patient_case_summary.head(20)
)

STUDY ↔ PATIENT RELATIONSHIP
Unique patients: 2224
Patients with multiple studies: 11


,n_cases,studies
patient_id,,
100047,5,5
100265,2,2
100455,2,2
100416,2,2
100268,2,2
100417,2,2
100456,2,2
101361,2,2
101595,2,2


In [8]:
# ============================================================
# CELL 8 — ANNOTATION / LABEL SOURCE AUDIT
# ============================================================

print("=" * 70)
print("ANNOTATION SOURCE AUDIT")
print("=" * 70)

for column in [
    "label",
    "diagnosis_source",
]:

    if column not in metadata.columns:
        print(
            f"\n{column}: column not found"
        )
        continue

    print(f"\n{column}:")
    print(
        metadata[column]
        .value_counts(dropna=False)
    )

ANNOTATION SOURCE AUDIT

label:
label
NaN    2238
Name: count, dtype: int64

diagnosis_source:
diagnosis_source
radiology           1212
histopathology       353
pathology            210
MSD_dataset          194
cytology             183
NIH_dataset           80
radiology / 3yFU       6
Name: count, dtype: int64


In [9]:
# ============================================================
# CELL 9 — MASK LABEL DISTRIBUTION
# ============================================================

print("=" * 70)
print("MASK LABEL DISTRIBUTION")
print("=" * 70)

if "mask_labels" not in metadata.columns:
    raise ValueError(
        "mask_labels column is missing."
    )

label_presence = {}

for labels_text in metadata["mask_labels"].astype(str):

    # Metadata entries were stored as strings.
    # Convert common formats into integer labels.

    cleaned = (
        labels_text
        .replace("[", "")
        .replace("]", "")
        .replace(",", " ")
    )

    labels = {
        int(x)
        for x in cleaned.split()
        if x.strip()
    }

    for label in labels:
        label_presence[label] = (
            label_presence.get(label, 0)
            + 1
        )

label_presence_df = (
    pd.DataFrame(
        [
            {
                "label": label,
                "cases_containing_label": count,
                "percentage": (
                    100 * count / len(metadata)
                ),
            }
            for label, count
            in sorted(label_presence.items())
        ]
    )
)

display(
    label_presence_df
)

MASK LABEL DISTRIBUTION


,label,cases_containing_label,percentage
0,0,2238,100.000000
1,1,676,30.205541
2,2,2238,100.000000
3,3,2238,100.000000
4,4,2238,100.000000
5,5,1778,79.445934
6,6,2022,90.348525


In [10]:
# ============================================================
# CELL 10 — PATIENT SEX DISTRIBUTION
# ============================================================

if "patient_sex" in metadata.columns:

    print("=" * 70)
    print("PATIENT SEX DISTRIBUTION")
    print("=" * 70)

    print(
        metadata["patient_sex"]
        .value_counts(dropna=False)
    )

PATIENT SEX DISTRIBUTION
patient_sex
M      1064
F       900
NaN     274
Name: count, dtype: int64


In [11]:
# ============================================================
# CELL 11 — PATIENT AGE DISTRIBUTION
# ============================================================

if "patient_age" in metadata.columns:

    ages = pd.to_numeric(
        metadata["patient_age"],
        errors="coerce"
    )

    print("=" * 70)
    print("PATIENT AGE DISTRIBUTION")
    print("=" * 70)

    print(
        "Missing ages:",
        int(ages.isna().sum())
    )

    print(
        "Minimum age:",
        ages.min()
    )

    print(
        "Maximum age:",
        ages.max()
    )

    print(
        "Mean age:",
        ages.mean()
    )

    print(
        "Median age:",
        ages.median()
    )

    print(
        "\nDescriptive statistics:"
    )

    display(
        ages.describe()
    )

PATIENT AGE DISTRIBUTION
Missing ages: 274
Minimum age: 18.0
Maximum age: 99.0
Mean age: 64.80957230142566
Median age: 67.0

Descriptive statistics:


count    1964.000000
mean       64.809572
std        13.685962
min        18.000000
25%        58.000000
50%        67.000000
75%        75.000000
max        99.000000
Name: patient_age, dtype: float64

In [12]:
# ============================================================
# CELL 12 — SCANNER DISTRIBUTION
# ============================================================

if "scanner" in metadata.columns:

    print("=" * 70)
    print("SCANNER DISTRIBUTION")
    print("=" * 70)

    scanner_counts = (
        metadata["scanner"]
        .value_counts(dropna=False)
    )

    print(
        "Unique scanner entries:",
        metadata["scanner"]
        .nunique(dropna=True)
    )

    display(
        scanner_counts.head(30)
    )

SCANNER DISTRIBUTION
Unique scanner entries: 6


scanner
SIEMENS                  938
TOSHIBA                  651
Philips                  319
NaN                      276
GE MEDICAL SYSTEMS        46
Canon Medical Systems      7
0                          1
Name: count, dtype: int64

In [13]:
# ============================================================
# CELL 13 — DIAGNOSIS DISTRIBUTION
# ============================================================

if "diagnosis" in metadata.columns:

    print("=" * 70)
    print("DIAGNOSIS DISTRIBUTION")
    print("=" * 70)

    diagnosis_counts = (
        metadata["diagnosis"]
        .value_counts(dropna=False)
    )

    print(
        "Unique diagnoses:",
        metadata["diagnosis"]
        .nunique(dropna=True)
    )

    display(
        diagnosis_counts.head(30)
    )

DIAGNOSIS DISTRIBUTION
Unique diagnoses: 2


diagnosis
non-PDAC    1562
PDAC         676
Name: count, dtype: int64

In [14]:
# ============================================================
# CELL 14 — METADATA ↔ FILE CONSISTENCY
# ============================================================

print("=" * 70)
print("METADATA ↔ FILE CONSISTENCY")
print("=" * 70)

missing_images = []
missing_masks = []

for study_id in study_ids:

    image_path = (
        IMAGES_DIR
        / f"{study_id}.npy"
    )

    mask_path = (
        MASKS_DIR
        / f"{study_id}.npy"
    )

    if not image_path.exists():
        missing_images.append(study_id)

    if not mask_path.exists():
        missing_masks.append(study_id)

print(
    "Metadata cases:",
    len(study_ids)
)

print(
    "Missing image files:",
    len(missing_images)
)

print(
    "Missing mask files:",
    len(missing_masks)
)

assert len(missing_images) == 0
assert len(missing_masks) == 0

print(
    "\n✓ Every metadata case has image + mask files."
)

METADATA ↔ FILE CONSISTENCY
Metadata cases: 2238
Missing image files: 0
Missing mask files: 0

✓ Every metadata case has image + mask files.


In [15]:
# ============================================================
# CELL 15 — FINAL AUDIT SUMMARY
# ============================================================

print("=" * 70)
print("FINAL DATASET AUDIT SUMMARY")
print("=" * 70)

print(
    "Total cases                :",
    len(metadata)
)

print(
    "Unique study IDs           :",
    study_ids.nunique()
)

print(
    "Unique patients            :",
    unique_patients
)

print(
    "Patients with >1 case      :",
    int(
        (cases_per_patient > 1)
        .sum()
    )
)

print(
    "Missing image files        :",
    len(missing_images)
)

print(
    "Missing mask files         :",
    len(missing_masks)
)

if "patient_sex" in metadata.columns:

    print(
        "Missing patient sex       :",
        int(
            metadata["patient_sex"]
            .isna()
            .sum()
        )
    )

if "patient_age" in metadata.columns:

    print(
        "Missing patient age       :",
        int(
            pd.to_numeric(
                metadata["patient_age"],
                errors="coerce"
            ).isna().sum()
        )
    )

print("\n" + "=" * 70)
print("DATASET AUDIT COMPLETE")
print("=" * 70)

FINAL DATASET AUDIT SUMMARY
Total cases                : 2238
Unique study IDs           : 2238
Unique patients            : 2224
Patients with >1 case      : 11
Missing image files        : 0
Missing mask files         : 0
Missing patient sex       : 274
Missing patient age       : 274

DATASET AUDIT COMPLETE


In [16]:
# ============================================================
# CELL 16 — LABEL SEMANTICS / DIAGNOSIS CROSS-AUDIT
# ============================================================

print("=" * 70)
print("LABEL PRESENCE × DIAGNOSIS AUDIT")
print("=" * 70)

# Parse mask_labels into sets
def parse_labels(value):
    cleaned = (
        str(value)
        .replace("[", "")
        .replace("]", "")
        .replace(",", " ")
    )
    return {
        int(x)
        for x in cleaned.split()
        if x.strip()
    }

label_sets = metadata["mask_labels"].apply(
    parse_labels
)

for label in sorted(
    set().union(*label_sets)
):
    metadata[f"has_label_{label}"] = (
        label_sets.apply(
            lambda s, lab=label: lab in s
        )
    )

# ------------------------------------------------------------
# Cross-tab with diagnosis
# ------------------------------------------------------------

for label in sorted(
    set().union(*label_sets)
):

    column = f"has_label_{label}"

    print("\n" + "-" * 70)
    print(f"LABEL {label} × DIAGNOSIS")
    print("-" * 70)

    print(
        pd.crosstab(
            metadata["diagnosis"],
            metadata[column],
            margins=True,
        )
    )

LABEL PRESENCE × DIAGNOSIS AUDIT

----------------------------------------------------------------------
LABEL 0 × DIAGNOSIS
----------------------------------------------------------------------
has_label_0  True   All
diagnosis              
PDAC          676   676
non-PDAC     1562  1562
All          2238  2238

----------------------------------------------------------------------
LABEL 1 × DIAGNOSIS
----------------------------------------------------------------------
has_label_1  False  True   All
diagnosis                     
PDAC             0   676   676
non-PDAC      1562     0  1562
All           1562   676  2238

----------------------------------------------------------------------
LABEL 2 × DIAGNOSIS
----------------------------------------------------------------------
has_label_2  True   All
diagnosis              
PDAC          676   676
non-PDAC     1562  1562
All          2238  2238

----------------------------------------------------------------------
LABEL 3 × D

In [17]:
# ============================================================
# CELL 17 — LABEL PRESENCE BY BATCH
# ============================================================

# Infer batch from study ID ranges currently used in the project.
# This is exploratory only; we are not modifying metadata.

def classify_batch(study_id):
    study_id = str(study_id)

    if study_id == "100936_00001":
        return "Batch 2 repaired"

    numeric_id = int(
        study_id.split("_")[0]
    )

    if numeric_id < 100565:
        return "Batch 1"

    if numeric_id < 101691:
        return "Batch 2"

    if numeric_id < 102224:
        return "Batch 3"

    return "Batch 4+"

metadata["audit_batch"] = (
    metadata["study_id"]
    .apply(classify_batch)
)

print("=" * 70)
print("LABEL PRESENCE BY BATCH")
print("=" * 70)

label_columns = [
    column
    for column in metadata.columns
    if column.startswith("has_label_")
]

batch_label_summary = (
    metadata
    .groupby("audit_batch")[label_columns]
    .mean()
    * 100
)

display(
    batch_label_summary
)

LABEL PRESENCE BY BATCH


,has_label_0,has_label_1,has_label_2,has_label_3,has_label_4,has_label_5,has_label_6
audit_batch,,,,,,,
Batch 1,100.0,28.347826,100.0,100.0,100.0,81.391304,92.000000
Batch 2,100.0,30.257320,100.0,100.0,100.0,79.503106,88.553682
Batch 2 repaired,100.0,100.000000,100.0,100.0,100.0,100.000000,100.000000
Batch 3,100.0,31.962617,100.0,100.0,100.0,77.196262,92.336449
